In [1]:
CITY    = "bologna"          

#=======================================
import warnings
warnings.filterwarnings("ignore")
import os, sys, tempfile, subprocess, glob
import numpy as np
import pandas as pd
import geopandas as gpd
import rasterio
import rasterio.features
from rasterio.transform import from_origin, rowcol
from rasterio.enums import Resampling
from rasterio.merge import merge
from rasterio.warp import reproject, Resampling, calculate_default_transform
from rasterio.mask import mask
from scipy import ndimage
from scipy.ndimage import label
from shapely.geometry import box, Point, shape, mapping
from shapely.ops import unary_union
from osgeo import gdal
from tqdm import tqdm
from scipy.ndimage import maximum_filter
from scipy.ndimage import watershed_ift
from shapely.strtree import STRtree
from shapely.geometry import box
from pathlib import Path
import numpy as np
import os
import pandas as pd

RESULTS_DIR = Path("/tmp/viewshed_chunks")
RESULTS_DIR.mkdir(exist_ok=True)
CHUNK_SIZE_M = 500

MIN_HEIGHT_M = 1.5
MIN_SCORE    = 0.3
MERGE_DIST_M = 3.0
HEIGHT_SIM   = 0.20

FLOOR_HEIGHT_M        = 3.0    # height of one floor (used to derive observer eye level)
OBSERVER_HEIGHT_M     = 1.6    # eye level above ground at civico point
TARGET_HEIGHT_M       = 1.5    # minimum tree height above ground that counts as visible
DEFAULT_TREE_HEIGHT_M = 5.0    # fallback when no LiDAR is available
BUFFER_M              = 30.0   # max visibility radius (metres) — controls GDAL viewshed extent
COMPLIANCE_THRESHOLD  = 3      # trees visible from a civico address to be "compliant"
VIEWSHED_CELL_M       = 1.0    # raster cell size for temporary viewshed rasters (metres)
AVG_BUILDING_HEIGHT_M = 10.0   # fallback building height when not available in civico data

ROOT_PATH = "/Users/terezasaskova/Library/Mobile Documents/com~apple~CloudDocs/3-30-300-index"
os.chdir(ROOT_PATH)
from config.config import (EUCLID_CUTOFF, connect_db, gdf_to_duckdb, load_table, gpkg_to_duckdb,get_city_cfg, get_output_paths,
        DB_PATH, GH_PORT, CRS_WGS, CRS_METRIC, WALKING_DISTANCE,
)
con = connect_db()

cfg     = get_city_cfg(CITY)
src     = cfg["data_sources"]
paths   = get_output_paths(CITY)

CRS = CRS_METRIC  # e.g. EPSG:32632 for Bologna


print(f"City: {CITY}")
print(f"DB path: {DB_PATH}")

trees = gpd.read_file("data/bologna/predictions_trees.gpkg").to_crs(CRS_METRIC)
gdf_to_duckdb(con, trees, f"raw.tree_polygons_{CITY}")

#wcs_url = "WCS:https://servizigis.regione.emilia-romagna.it/wcs/dtmrer2023_24"
#dtm_bologna = gdal.Open(wcs_url)

City: bologna
DB path: /Users/terezasaskova/3_30_300_master.duckdb


FloatProgress(value=0.0, layout=Layout(width='auto'), style=ProgressStyle(bar_color='black'))

raw.tree_polygons_bologna: 219955 rows written


In [2]:
boundary = load_table(con, "raw", f"{CITY}_boundary")
buildings = load_table(con, "raw", f"{CITY}_buildings")   # must have 'height' column  - from gba
civici = load_table(con, "raw", f"{CITY}_buildings_sn")
trees = load_table(con, "raw", f"tree_polygons_{CITY}")  # crown polygons + score + height_m
#dsm_bologna = rasterio.open(f"data/bologna/dsm_bologna.tif")

buildings = buildings.to_crs(CRS)
civici = civici.to_crs(CRS)
trees = trees.to_crs(CRS)

print(f"buildings: {len(buildings):,}  |  civici: {len(civici):,}  |  trees: {len(trees):,}")

trees["source"] = "deepforest"
trees = trees[trees["score"] >= MIN_SCORE].copy()
trees = trees.reset_index(drop=True)
print(f"Deepforest trees after filtering: {len(trees):,}")


# ── prep ──────────────────────────────────────────────────────────────────────
trees["centroid"] = trees.geometry.centroid
trees["tree_id"]  = trees.index
tree_centroids_gs = gpd.GeoSeries(trees["centroid"], crs=CRS)
print(f"Tree centroids ready: {len(tree_centroids_gs):,}")



buildings: 36,006  |  civici: 77,630  |  trees: 219,955
Deepforest trees after filtering: 219,955
Tree centroids ready: 219,955


## GDAL viewshed helper

`gdal_viewshed` (GDAL ≥ 3.1) computes a binary visible/not-visible raster  
from a single observer point.  We call the CLI via `subprocess` — no Python  
binding needed, and it is much faster than a pure-Python ray-cast loop.

In [3]:
def gdal_viewshed_visible_trees(
    observer_x: float,
    observer_y: float,
    dsm_path: str,
    tree_centroids: gpd.GeoSeries,
    observer_z: float = 1.5,       # renamed from observer_height
    target_height: float = TARGET_HEIGHT_M,
    max_distance: float = BUFFER_M,
) -> np.ndarray:

    out_tif = tempfile.NamedTemporaryFile(suffix=".tif", delete=False).name
    try:
        cmd = [
            "gdal_viewshed",
            "-ox",  str(observer_x),
            "-oy",  str(observer_y),
            "-oz",  str(observer_z),
            "-tz",  str(target_height),
            "-md",  str(max_distance),
            "-f",   "GTiff",
            "-vv",  "255",   # visible pixel value
            "-iv",  "0",     # invisible pixel value
            "-ov",  "0",     # out-of-range value
            dsm_path,
            out_tif,
        ]
        try:
            subprocess.run(cmd, check=True, capture_output=True)
        except subprocess.CalledProcessError as e:
            return np.zeros(len(tree_centroids), dtype=bool)

        with rasterio.open(out_tif) as vs:
            vs_data      = vs.read(1)
            vs_transform = vs.transform

        # Sample the viewshed raster at each tree centroid
        visible = np.zeros(len(tree_centroids), dtype=bool)
        for i, pt in enumerate(tree_centroids):
            try:
                r, c = rowcol(vs_transform, pt.x, pt.y)
                if 0 <= r < vs_data.shape[0] and 0 <= c < vs_data.shape[1]:
                    visible[i] = vs_data[r, c] == 255
            except Exception:
                pass
        return visible

    finally:
        if os.path.exists(out_tif):
            os.remove(out_tif)


print("gdal_viewshed helper defined")

gdal_viewshed helper defined


In [12]:
import numpy as np
import rasterio
from rasterio.transform import from_bounds
from rasterio.features import rasterize
from shapely.geometry import mapping

RESOLUTION_M = 0.5

def buildings_to_dsm(buildings: gpd.GeoDataFrame, output_path: str, resolution: float = RESOLUTION_M) -> str:
    """Rasterize building footprints+heights into a DSM GeoTIFF."""
    
    buildings = buildings[buildings.geometry.notna() & buildings.geometry.is_valid]
    if buildings.empty:
            return None 
    minx, miny, maxx, maxy = buildings.total_bounds
    
    # add buffer so edge buildings aren't clipped
    pad = 50
    minx -= pad; miny -= pad; maxx += pad; maxy += pad

    width  = int((maxx - minx) / resolution)
    height = int((maxy - miny) / resolution)
    transform = from_bounds(minx, miny, maxx, maxy, width, height)

    shapes = (
        (mapping(geom), h)
        for geom, h in zip(buildings.geometry, buildings["height"])
        if geom is not None and h > 0
    )

    dsm = rasterize(
        shapes,
        out_shape=(height, width),
        transform=transform,
        fill=0.0,        # ground level where no building
        dtype="float32",
        merge_alg=rasterio.enums.MergeAlg.replace,
    )

    with rasterio.open(
        output_path,
        "w",
        driver="GTiff",
        height=height,
        width=width,
        count=1,
        dtype="float32",
        crs=buildings.crs,
        transform=transform,
    ) as dst:
        dst.write(dsm, 1)

    print(f"DSM written: {output_path}  ({width}x{height} px)")
    return output_path


In [5]:
DSM_PATH = "/tmp/buildings_dsm.tif"
buildings_to_dsm(buildings, DSM_PATH)

DSM written: /tmp/buildings_dsm.tif  (33490x31118 px)


'/tmp/buildings_dsm.tif'

## Viewshed loop: one row per (civico × floor) combination


In [ ]:
OBSERVER_HEIGHT_M = 1.6
tree_idx   = STRtree(tree_centroids_gs.values)
tree_array = tree_centroids_gs.values
tree_ids   = trees["tree_id"].values

civici = civici.drop(columns=["index_right"], errors="ignore")
civici = gpd.sjoin(civici, buildings[["height", "geometry"]], how="left", predicate="within")
civici = civici.drop_duplicates(subset="civkey")
keep_cols = ["civkey", "indirizzo_completo", "building_id", "geometry", "height_right"]
civici = civici[keep_cols].rename(columns={"height_right": "height"})
civici["observer_z"] = OBSERVER_HEIGHT_M
civici["height"] = civici["height"].fillna(AVG_BUILDING_HEIGHT_M).replace(0, AVG_BUILDING_HEIGHT_M)
civici["floor_count"] = (civici["height"] / FLOOR_HEIGHT_M).astype(int).clip(lower=1)

def process_civico(civ, tree_idx, tree_array, tree_ids, dsm_path):
    ox, oy         = civ.geometry.x, civ.geometry.y
    candidate_idxs = tree_idx.query(civ.geometry.buffer(BUFFER_M))
    n_nearby       = len(candidate_idxs)

    if n_nearby > 0:
        candidate_centroids = gpd.GeoSeries(tree_array[candidate_idxs], crs=CRS)
        candidate_tree_ids  = tree_ids[candidate_idxs]

    base_row = {
        "civkey":             civ.civkey,
        "indirizzo_completo": civ.indirizzo_completo, #adress 
        "building_id":        civ.building_id,
        "geometry":           civ.geometry,
    }

    rows = []
    for floor_idx in range(civ.floor_count):
        observer_z = (floor_idx * FLOOR_HEIGHT_M) + 1.5

        if n_nearby == 0:
            rows.append({**base_row, "floor": floor_idx, "observer_z_m": observer_z,
                         "n_trees_nearby": 0, "n_trees_visible": 0,
                         "visible_tree_ids": [], "compliant": False})
            continue

        visible_mask = gdal_viewshed_visible_trees(
            observer_x=ox, observer_y=oy, observer_z=observer_z,
            tree_centroids=candidate_centroids, dsm_path=dsm_path,
        )
        visible_ids = candidate_tree_ids[visible_mask].tolist()
        n_visible   = int(visible_mask.sum())

        rows.append({**base_row, "floor": floor_idx, "observer_z_m": observer_z,
                     "n_trees_nearby": n_nearby, "n_trees_visible": n_visible,
                     "visible_tree_ids": visible_ids,
                     "compliant": n_visible >= COMPLIANCE_THRESHOLD})
    return rows

#chunk loop 
minx, miny, maxx, maxy = civici.total_bounds
chunk_id = 0

for x0 in tqdm(np.arange(minx, maxx, CHUNK_SIZE_M), desc="Chunks"):
    for y0 in np.arange(miny, maxy, CHUNK_SIZE_M):
        tile           = box(x0, y0, x0 + CHUNK_SIZE_M, y0 + CHUNK_SIZE_M)
        civici_tile    = civici[civici.geometry.within(tile)]
        buildings_tile = buildings[buildings.geometry.intersects(tile.buffer(BUFFER_M))]

        if civici_tile.empty:
            continue

        tile_dsm = f"/tmp/dsm_{chunk_id}.tif"
        dsm_result = buildings_to_dsm(buildings_tile, tile_dsm)
        if dsm_result is None:
            chunk_id += 1
            continue

        rows = []
        for civ in civici_tile.itertuples():
            rows.extend(process_civico(civ, tree_idx, tree_array, tree_ids, tile_dsm))

        if rows:
            chunk_gdf = gpd.GeoDataFrame(rows, geometry="geometry", crs=CRS)
            chunk_gdf.to_parquet(RESULTS_DIR / f"chunk_{chunk_id:04d}.parquet")

        os.remove(tile_dsm)
        chunk_id += 1

print(f"Chunks written: {chunk_id}")

#save 
all_chunks = sorted(RESULTS_DIR.glob("*.parquet"))
result_gdf = gpd.GeoDataFrame(
    pd.concat([gpd.read_parquet(f) for f in all_chunks]),
    geometry="geometry", crs=CRS_METRIC
)

# compliance summary per civico across all floors
civico_compliance = (
    result_gdf.groupby("civkey")["compliant"]
    .any()
    .rename("any_floor_compliant")
    .reset_index()
)
result_gdf = result_gdf.merge(civico_compliance, on="civkey", how="left")

# rename
rename = {"indirizzo_completo": "address"}
if cfg.get("neighbourhood_col") and cfg["neighbourhood_col"] in result_gdf.columns:
    rename[cfg["neighbourhood_col"]] = "neighbourhood"
result_gdf = result_gdf.rename(columns=rename)

# WGS84
out_wgs = result_gdf.to_crs(CRS_WGS)
out_wgs["longitude"] = out_wgs.geometry.x
out_wgs["latitude"]  = out_wgs.geometry.y

final_cols = [
    "civkey", "address", "longitude", "latitude", "building_id",
    "floor", "observer_z_m", "n_trees_nearby", "n_trees_visible",
    "visible_tree_ids", "compliant", "any_floor_compliant", "geometry",
]
out_wgs = out_wgs[[c for c in final_cols if c in out_wgs.columns]]

# DuckDB
con = connect_db()
gdf_to_duckdb(con, result_gdf, f"rule_3.{CITY}_results")

# GeoPackage
paths = get_output_paths(CITY)
paths["gpkg"] = paths["gpkg"].replace(".gpkg", "_rule3.gpkg")
result_gdf.to_file(paths["gpkg"], driver="GPKG", layer="civici")

# Parquet + view
parquet_path = paths["parquet"].replace(".parquet", "_rule3.parquet")
out_wgs.to_parquet(parquet_path, index=False)
con.execute(f"""
    CREATE OR REPLACE VIEW rule_3.{CITY}_geoparquet AS
    SELECT * FROM read_parquet('{parquet_path}')
""")
print(f"{CITY} rule_3 view registered")
con.close()

Chunks:   0%|          | 0/33 [00:00<?, ?it/s]

DSM written: /tmp/dsm_0.tif  (1229x1598 px)
DSM written: /tmp/dsm_1.tif  (1183x1236 px)
DSM written: /tmp/dsm_2.tif  (728x454 px)
DSM written: /tmp/dsm_3.tif  (1313x1048 px)
DSM written: /tmp/dsm_4.tif  (1513x1123 px)


Chunks:   3%|▎         | 1/33 [00:03<01:44,  3.28s/it]

DSM written: /tmp/dsm_5.tif  (1003x1011 px)
DSM written: /tmp/dsm_6.tif  (1431x1255 px)
DSM written: /tmp/dsm_7.tif  (1394x1190 px)
DSM written: /tmp/dsm_8.tif  (1370x1160 px)
DSM written: /tmp/dsm_9.tif  (1125x1276 px)
DSM written: /tmp/dsm_10.tif  (1322x1281 px)
DSM written: /tmp/dsm_11.tif  (1085x1179 px)
DSM written: /tmp/dsm_12.tif  (1376x1315 px)
DSM written: /tmp/dsm_13.tif  (1367x1391 px)
DSM written: /tmp/dsm_14.tif  (1331x981 px)
DSM written: /tmp/dsm_15.tif  (840x700 px)


Chunks:   6%|▌         | 2/33 [00:06<01:35,  3.08s/it]

...

DSM written: /tmp/dsm_178.tif  (684x739 px)
DSM written: /tmp/dsm_179.tif  (1010x777 px)
DSM written: /tmp/dsm_180.tif  (853x1209 px)
DSM written: /tmp/dsm_181.tif  (1096x1097 px)
DSM written: /tmp/dsm_182.tif  (1377x652 px)
DSM written: /tmp/dsm_183.tif  (1377x1331 px)
DSM written: /tmp/dsm_184.tif  (612x644 px)
DSM written: /tmp/dsm_185.tif  (657x1276 px)
DSM written: /tmp/dsm_186.tif  (1320x1327 px)
DSM written: /tmp/dsm_187.tif  (1315x1210 px)
DSM written: /tmp/dsm_188.tif  (951x1273 px)
DSM written: /tmp/dsm_189.tif  (1224x1359 px)
DSM written: /tmp/dsm_190.tif  (1426x1412 px)
DSM written: /tmp/dsm_191.tif  (1635x1390 px)
DSM written: /tmp/dsm_192.tif  (1488x1612 px)
DSM written: /tmp/dsm_193.tif  (1505x1493 px)
DSM written: /tmp/dsm_194.tif  (1420x1424 px)
DSM written: /tmp/dsm_195.tif  (1833x1442 px)
DSM written: /tmp/dsm_196.tif  (1819x1470 px)
DSM written: /tmp/dsm_197.tif  (1489x1409 px)
DSM written: /tmp/dsm_198.tif  (1552x1383 px)
DSM written: /tmp/dsm_199.tif  (1472x1558 p

Chunks:  42%|████▏     | 14/33 [3:06:48<7:48:17, 1478.84s/it]

DSM written: /tmp/dsm_203.tif  (600x1247 px)
DSM written: /tmp/dsm_204.tif  (1119x963 px)
DSM written: /tmp/dsm_205.tif  (1090x803 px)
DSM written: /tmp/dsm_206.tif  (807x1303 px)
DSM written: /tmp/dsm_207.tif  (1115x1340 px)
DSM written: /tmp/dsm_208.tif  (715x1291 px)
DSM written: /tmp/dsm_209.tif  (1081x1225 px)
DSM written: /tmp/dsm_210.tif  (885x1402 px)
DSM written: /tmp/dsm_211.tif  (985x1273 px)
DSM written: /tmp/dsm_212.tif  (1253x1351 px)
DSM written: /tmp/dsm_213.tif  (1254x1096 px)
DSM written: /tmp/dsm_214.tif  (1129x1182 px)
DSM written: /tmp/dsm_215.tif  (1338x1312 px)
DSM written: /tmp/dsm_216.tif  (1435x1450 px)
DSM written: /tmp/dsm_217.tif  (1408x1390 px)
DSM written: /tmp/dsm_218.tif  (1482x1530 px)
DSM written: /tmp/dsm_219.tif  (1724x1886 px)
DSM written: /tmp/dsm_220.tif  (1622x1344 px)
DSM written: /tmp/dsm_221.tif  (1729x1572 px)
DSM written: /tmp/dsm_222.tif  (1540x1391 px)
DSM written: /tmp/dsm_223.tif  (1447x1403 px)
DSM written: /tmp/dsm_224.tif  (1332x1278

Chunks:  45%|████▌     | 15/33 [3:41:09<8:16:14, 1654.15s/it]

DSM written: /tmp/dsm_227.tif  (1137x1232 px)
DSM written: /tmp/dsm_228.tif  (1092x1334 px)
DSM written: /tmp/dsm_229.tif  (870x1054 px)
DSM written: /tmp/dsm_230.tif  (588x482 px)
DSM written: /tmp/dsm_231.tif  (918x559 px)
DSM written: /tmp/dsm_232.tif  (326x439 px)
DSM written: /tmp/dsm_233.tif  (329x953 px)
DSM written: /tmp/dsm_234.tif  (749x1078 px)
DSM written: /tmp/dsm_235.tif  (1310x1072 px)
DSM written: /tmp/dsm_236.tif  (1347x1282 px)
DSM written: /tmp/dsm_237.tif  (1362x688 px)
DSM written: /tmp/dsm_238.tif  (1172x759 px)
DSM written: /tmp/dsm_239.tif  (1316x1175 px)
DSM written: /tmp/dsm_240.tif  (1381x1326 px)
DSM written: /tmp/dsm_241.tif  (1518x1258 px)
DSM written: /tmp/dsm_242.tif  (1857x1468 px)
DSM written: /tmp/dsm_243.tif  (1837x1723 px)
DSM written: /tmp/dsm_244.tif  (1826x1797 px)
